# engineeringkit quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cpoli/engineeringkit/blob/main/notebooks/quickstart.ipynb)

engineeringkit makes the models of twelve engineering domains computable and inspectable. Inputs and outputs are
plain numbers and NumPy arrays in SI units, solvers return small result objects with `.summary()` and `.plot()`,
and every function validates its inputs. Ten minutes, six domains.

In [ ]:
%pip install -q engineeringkit

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import engineeringkit as ek

print(ek.__version__)

## 1. Structures: a beam under a point load

A 5 m steel beam ($E = 200$ GPa, $I = 8\times10^{-6}$ m$^4$) with 10 kN at midspan. The result holds the shear,
moment and deflection along the beam; the midspan deflection matches $PL^3/(48EI)$.

In [ ]:
from engineeringkit.structural_engineering import SimplySupportedBeam

L, E, I, P = 5.0, 200e9, 8e-6, 10e3
result = SimplySupportedBeam(L, E, I).solve_point_load(P)
print(result.summary())
print("PL^3/(48EI) =", P * L**3 / (48 * E * I), "m")
fig, axes = result.plot()
for ax, label in zip(axes, ["shear [N]", "moment [N m]", "deflection [m]"]):
    ax.set_ylabel(label)

## 2. Vibration: three amounts of damping

A 1 kg mass on a 4 N/m spring, released from 0.1 m. `simulate_sdof` integrates $m\ddot x + c\dot x + kx = 0$.

In [ ]:
from engineeringkit.mechanical_engineering import damping_ratio, simulate_sdof

t = np.linspace(0, 10, 501)
fig, ax = plt.subplots()
for c in (0.4, 1.6, 4.0):
    sdof = simulate_sdof(1.0, c, 4.0, x0=0.1, t=t)
    ax.plot(sdof.t, sdof.displacement, label=f"zeta = {damping_ratio(c, 4.0, 1.0):.1f}")
ax.set(xlabel="time [s]", ylabel="displacement [m]")
ax.legend();

## 3. Fluids: the Moody diagram

Colebrook's friction factor for turbulent pipe flow, with the laminar $64/Re$ line, for three relative roughnesses.

In [ ]:
from engineeringkit.fluid_engineering import colebrook_friction_factor, laminar_friction_factor

fig, ax = plt.subplots()
laminar = np.logspace(2.8, np.log10(2299), 50)
ax.loglog(laminar, [laminar_friction_factor(re) for re in laminar], "k")
turbulent = np.logspace(np.log10(4000), 8, 100)
for roughness in (0.0, 1e-4, 1e-2):
    f = [colebrook_friction_factor(re, roughness) for re in turbulent]
    ax.loglog(turbulent, f, label=f"relative roughness {roughness:g}")
ax.set(xlabel="Reynolds number", ylabel="Darcy friction factor")
ax.legend();

## 4. Control: a Ziegler-Nichols PID

The plant $1/(s+1)^3$ oscillates under proportional control at the gain margin $K_u = 8$. Ziegler and Nichols's
rules turn $K_u$ and the oscillation period into PID gains.

In [ ]:
from engineeringkit import control_systems as cs

G = cs.transfer_function([1], [1, 3, 3, 1])
margins = cs.stability_margins(G)
ku, tu = margins["gain_margin"], 2 * np.pi / margins["phase_cross_frequency"]
pid = cs.ziegler_nichols(ku, tu)
print(f"K_u = {ku:.2f}, T_u = {tu:.2f} s ->", pid)
response = pid.simulate(G, T=np.linspace(0, 30, 1001))
fig, ax = response.plot()
ax.set(xlabel="time [s]", ylabel="output");

## 5. Communications: bit errors against the Shannon limit

The bit error rate of BPSK in white Gaussian noise, and the $-1.59$ dB below which no code can work.

In [ ]:
from engineeringkit.communications import ber_bpsk_awgn, db, from_db

ebn0_db = np.linspace(-2, 12, 141)
fig, ax = plt.subplots()
ax.semilogy(ebn0_db, ber_bpsk_awgn(from_db(ebn0_db)), label="BPSK")
ax.axvline(db(np.log(2)), color="k", ls="--", label="Shannon limit")
ax.set(xlabel="Eb/N0 [dB]", ylabel="bit error rate", ylim=(1e-8, 1))
ax.legend();

## 6. Reliability: three Weibull shapes

The Weibull shape $\beta$ says whether a population fails young ($\beta < 1$), at random ($\beta = 1$) or by
wear-out ($\beta > 1$).

In [ ]:
from engineeringkit.reliability import weibull_mttf, weibull_reliability

t = np.linspace(0, 3000, 301)
fig, ax = plt.subplots()
for beta in (0.5, 1.0, 3.0):
    ax.plot(t, weibull_reliability(t, 1000.0, beta), label=f"beta = {beta}, MTTF {weibull_mttf(1000.0, beta):.0f} h")
ax.set(xlabel="time [h]", ylabel="survival probability")
ax.legend();

## Where next

- The [tutorials](https://cpoli.github.io/engineeringkit/tutorials/index.html) follow one problem across several
  domains: a beam from load to fatigue life, a feedback loop in time and frequency, a power plant from cycle to
  cooling tower, and a radio link from antenna to bit errors.
- The [exercises](https://cpoli.github.io/engineeringkit/exercises/index.html) give a problem set per domain, with
  worked solutions.
- Every domain has a [gallery](https://cpoli.github.io/engineeringkit/examples/index.html) of runnable examples and a
  history page tracing its breakthroughs.